# 01 · Explore the Original TCGA-COAD Data

Walk through the raw data before any BOVIN processing.

**Source**: UCSC Xena · `TCGA.COAD.sampleMap` · log2(RSEM+1)

**Files** (gitignored, in `data/raw/`):
- `HiSeqV2`               — gene × sample expression matrix
- `COAD_clinicalMatrix`   — clinical covariates per patient
- `COAD_survival.txt`     — OS / DSS / DFI / PFI endpoints

Run from the repo root:
```bash
make docker-shell
# in container:
jupyter lab notebooks/
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

RAW = Path('../data/raw')
assert RAW.exists(), f'{RAW} not found — run `make data-coad` first'
print('files in', RAW)
for p in sorted(RAW.iterdir()):
    print(f'  {p.name:28s}  {p.stat().st_size/1024:.0f} KB')

## 1. RNA-seq matrix (`HiSeqV2`)

Xena convention: **rows = genes, columns = samples, values = log2(RSEM+1)**.

In [ ]:
expr = pd.read_csv(RAW / 'HiSeqV2', sep='\t', index_col=0)
expr.index.name = 'gene'
print(f'shape: {expr.shape}  ({expr.shape[0]} genes × {expr.shape[1]} samples)')
expr.iloc[:5, :5]

In [ ]:
# Sample barcodes are TCGA-XX-YYYY-ZZ where ZZ=01 primary tumor, ZZ=11 normal
barcode_suffix = expr.columns.str[-2:].value_counts()
print('Sample type suffix:\n', barcode_suffix)
print("\nA few real sample IDs:")
print(list(expr.columns[:5]))

In [ ]:
# Transpose for downstream: samples × genes
df = expr.T
df.index.name = 'sample'
print(f'patient × gene frame: {df.shape}')
df.iloc[:5, :5]

### 1.1 Distribution of expression values (all 329 samples × 20,530 genes)

The bimodal / long-tail shape is the classic log2(RSEM+1) signature — mass at 0 (unexpressed) + a broader hump from ~6 to ~14.

In [ ]:
flat = df.values.ravel()
print(f'N values = {flat.size:,}')
print(f'min={flat.min():.2f}  max={flat.max():.2f}  mean={flat.mean():.2f}  median={np.median(flat):.2f}')
print(f'% exactly 0 = {(flat==0).mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(flat, bins=80, color='#5B8DEF', alpha=0.85)
ax.set_xlabel('log2(RSEM+1)')
ax.set_ylabel('# gene × sample counts')
ax.set_title('TCGA-COAD RNA-seq · whole-matrix distribution')
plt.show()

## 2. The 6 label genes · distribution inspection

These are the genes in our ICD-readiness surrogate signature.

In [ ]:
POS = ['CALR', 'HMGB1', 'HSPA1A', 'HSP90AA1']
NEG = ['CD47', 'CD24']
LABEL_GENES = POS + NEG

assert all(g in df.columns for g in LABEL_GENES), 'some label genes missing!'
label_df = df[LABEL_GENES]
label_df.describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, g in zip(axes.ravel(), LABEL_GENES):
    color = '#F2A33A' if g in POS else '#9B59B6'
    sign = '+' if g in POS else '−'
    ax.hist(label_df[g], bins=40, color=color, alpha=0.85)
    ax.set_title(f'{sign} {g}   (n={df.shape[0]})', fontsize=11)
    ax.set_xlabel('log2(RSEM+1)')
    ax.axvline(label_df[g].mean(), color='k', ls='--', lw=0.8, label='mean')
    ax.axvline(label_df[g].median(), color='k', ls=':', lw=0.8, label='median')
    ax.legend(fontsize=8)
plt.suptitle('6 label genes — raw expression in TCGA-COAD', y=1.02)
plt.tight_layout()
plt.show()

### 2.1 Are the 4 DAMPs co-regulated? (Probe 1 of our surrogate validation)

In [ ]:
corr = label_df[POS].corr()
print(corr.round(3))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            fmt='.2f', ax=ax, cbar_kws={'label': 'Pearson r'})
ax.set_title(f'4 DAMP correlation — mean off-diag r = {(corr.sum().sum()-4)/(4*3):.2f}')
plt.show()

In [ ]:
# Full 6-gene correlation
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(label_df.corr(), annot=True, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            fmt='.2f', ax=ax)
ax.set_title('All 6 label genes — Pearson correlation')
plt.show()

## 3. Compute the surrogate signature yourself

This is literally what `bovin_demo.data.labels.icd_readiness_label` does.

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

signature = (
    zscore(df['CALR']) + zscore(df['HMGB1']) +
    zscore(df['HSPA1A']) + zscore(df['HSP90AA1'])
    - zscore(df['CD47']) - zscore(df['CD24'])
)
threshold = signature.median()
label = (signature > threshold).astype(int)

print(f'N patients = {len(signature)}')
print(f'signature range: [{signature.min():+.2f}, {signature.max():+.2f}]')
print(f'median threshold = {threshold:+.4f}')
print(f'label pos_rate = {label.mean():.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#EF5B5B' if v==0 else '#4FB286' for v in label]
ax.hist([signature[label==0], signature[label==1]],
        bins=40, stacked=True, color=['#EF5B5B', '#4FB286'],
        label=['COLD (0)', 'READY (1)'])
ax.axvline(threshold, color='k', ls='--', label=f'median={threshold:+.3f}')
ax.set_xlabel('surrogate signature score')
ax.set_ylabel('# patients')
ax.set_title('ICD-readiness surrogate · z-score composite split at median')
ax.legend()
plt.show()

## 4. Clinical + survival annotations

In [ ]:
clin = pd.read_csv(RAW / 'COAD_clinicalMatrix', sep='\t', index_col=0, low_memory=False)
print(f'clinical shape: {clin.shape}')
print(f'columns ({len(clin.columns)} total), first 30:')
for c in list(clin.columns)[:30]:
    print(f'  {c}')

In [ ]:
interesting = ['gender', 'age_at_initial_pathologic_diagnosis',
               'pathologic_stage', 'sample_type', 'MSI_updated_Oct62011']
available = [c for c in interesting if c in clin.columns]
clin[available].head()

In [ ]:
# Stage distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
if 'pathologic_stage' in clin.columns:
    clin['pathologic_stage'].value_counts().plot.barh(ax=axes[0])
    axes[0].set_title('pathologic_stage')
if 'age_at_initial_pathologic_diagnosis' in clin.columns:
    clin['age_at_initial_pathologic_diagnosis'].dropna().hist(bins=20, ax=axes[1], color='#5B8DEF')
    axes[1].set_title('age at diagnosis')
if 'gender' in clin.columns:
    clin['gender'].value_counts().plot.bar(ax=axes[2], color=['#F2A33A', '#9B59B6'])
    axes[2].set_title('gender')
plt.tight_layout()
plt.show()

In [ ]:
surv = pd.read_csv(RAW / 'COAD_survival.txt', sep='\t')
print(f'survival shape: {surv.shape}')
print(f'cols: {list(surv.columns)}')
surv.head()

In [ ]:
# Event counts per endpoint
print('Events per endpoint:')
for col in ['OS', 'DSS', 'DFI', 'PFI']:
    if col in surv.columns:
        n = surv[col].notna().sum()
        events = (surv[col] == 1).sum()
        print(f'  {col}: {events}/{n}  ({events/n*100:.1f}%)')

## 5. Join everything on sample id

The final per-patient data frame that the BOVIN pipeline uses.

In [ ]:
surv_indexed = surv.set_index('sample')
joined = pd.concat([
    label_df,                                    # 6 label genes
    pd.Series(signature, name='signature'),
    pd.Series(label, name='label'),
    surv_indexed[['PFI', 'PFI.time', 'OS', 'OS.time']],
    clin.get('pathologic_stage'),
    clin.get('age_at_initial_pathologic_diagnosis'),
], axis=1).dropna(subset=['label'])
print(f'joined shape: {joined.shape}')
joined.head()

### 5.1 Pick any patient and see everything about them

Edit `PATIENT_ID` to explore a specific case.

In [ ]:
PATIENT_ID = 'TCGA-CA-5256-01'   # the COLD example used in the demo walkthrough
# PATIENT_ID = 'TCGA-DM-A1DA-01' # most-confident READY example

row = joined.loc[PATIENT_ID]
print(f'=== {PATIENT_ID} ===\n')
print('6 label genes (log2(RSEM+1)):')
for g in LABEL_GENES:
    print(f'  {g:10s} = {row[g]:.4f}')
print(f'\nSurrogate signature = {row["signature"]:+.4f}')
print(f'Binary label        = {int(row["label"])} ({"READY" if row["label"]==1 else "COLD"})')
print(f'\nSurvival:')
print(f'  PFI event: {int(row["PFI"]) if pd.notna(row["PFI"]) else "na"}  PFI.time: {row["PFI.time"]:.0f} days')
print(f'  OS  event: {int(row["OS"])  if pd.notna(row["OS"])  else "na"}  OS.time:  {row["OS.time"]:.0f} days')
if 'pathologic_stage' in row:
    print(f'  stage: {row["pathologic_stage"]}')
if 'age_at_initial_pathologic_diagnosis' in row:
    print(f'  age:   {row["age_at_initial_pathologic_diagnosis"]}')

## 6. BOVIN-Pathway graph at a glance

The biology prior (82 nodes / 99 edges) that the GNN is constrained to.

In [ ]:
import json
with open('../bovin_demo/graph/bovin_pathway_v0.json') as f:
    graph = json.load(f)

print(f'nodes: {len(graph["nodes"])}')
print(f'edges: {len(graph["edges"])}')
print(f'modules: {[m["id"] for m in graph["modules"]]}')

# Nodes per module
nodes_df = pd.DataFrame(graph['nodes'])
print('\nnodes per module:')
print(nodes_df.groupby('module').size())

In [ ]:
print(f'observable genes (= accept RNA-seq input): {nodes_df["observable"].sum()} / {len(nodes_df)}')
print(f'\nnode types:')
print(nodes_df["type"].value_counts())

In [ ]:
# Which of the 70 observable genes are present in our TCGA matrix?
obs = nodes_df[nodes_df['observable']]
hit = obs['symbol'].isin(df.columns).sum()
print(f'hit rate: {hit}/{len(obs)}  ({hit/len(obs)*100:.1f}%)')
print('\nmisses (symbols in graph but not in Xena matrix):')
for s in obs.loc[~obs['symbol'].isin(df.columns), 'symbol']:
    print(f'  {s}')

## 7. Save-for-later checkpoints

These are the frames every downstream notebook / script will use.

In [ ]:
OUT = Path('../data/processed')
OUT.mkdir(exist_ok=True)

df.to_parquet(OUT / 'tcga_coad_expr.parquet')  # samples × genes
joined[['signature', 'label', 'PFI', 'PFI.time', 'OS', 'OS.time']].to_csv(
    OUT / 'tcga_coad_label_and_survival.csv'
)
print('wrote:')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:40s}  {p.stat().st_size/1024:.0f} KB')

---

**Next notebook**: `02_gene_alignment_and_heterodata.ipynb` (M2-M3 bridge)

See `bovin_demo/data/` and `docs/demo_card.md` for the full pipeline.